In [1]:
from pathlib import Path
import shutil
import random
from typing import Union, Tuple

def split_train_test(
    images_dir: Union[str, Path],
    train_ratio: float = 0.8,
    seed: int = 42
) -> Tuple[int, int]:
    """
    Verilen klasördeki görselleri train ve test klasörlerine ayırır.
    
    Args:
        images_dir: Görsellerin bulunduğu klasör path'i
        train_ratio: Train oranı (default: 0.8 = %80)
        seed: Rastgelelik için seed (reproducibility için)
    
    Returns:
        (train_count, test_count) - Train ve test klasörlerine taşınan resim sayıları
    """
    images_dir = Path(images_dir)
    
    if not images_dir.exists():
        raise FileNotFoundError(f"Klasör bulunamadı: {images_dir}")
    
    if train_ratio <= 0 or train_ratio >= 1:
        raise ValueError("train_ratio 0 ile 1 arasında olmalı")
    
    # Train ve test klasörlerini oluştur
    train_dir = images_dir / "train"
    test_dir = images_dir / "test"
    
    train_dir.mkdir(exist_ok=True)
    test_dir.mkdir(exist_ok=True)
    
    # Desteklenen resim formatları
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
    
    # images_dir içindeki dosyaları listele (train/test klasörleri hariç)
    image_files = [
        f for f in images_dir.iterdir()
        if f.is_file() and f.suffix.lower() in image_extensions
    ]
    
    if len(image_files) == 0:
        raise ValueError(f"Resim bulunamadı: {images_dir}")
    
    # Rastgeleliği sabitle (reproducibility)
    random.seed(seed)
    random.shuffle(image_files)
    
    # Train/test ayırımı
    split_idx = int(len(image_files) * train_ratio)
    train_files = image_files[:split_idx]
    test_files = image_files[split_idx:]
    
    print(f"Toplam resim: {len(image_files)}")
    print(f"Train oranı: {train_ratio*100:.1f}% ({len(train_files)} resim)")
    print(f"Test oranı: {(1-train_ratio)*100:.1f}% ({len(test_files)} resim)")
    print()
    
    # Train klasörüne kopyala/taşı
    print("Train klasörüne taşınıyor...")
    for i, src_file in enumerate(train_files, 1):
        dst_file = train_dir / src_file.name
        shutil.move(str(src_file), str(dst_file))
        if i % 100 == 0:
            print(f"  {i}/{len(train_files)}")
    print(f"✓ {len(train_files)} resim train klasörüne taşındı")
    print()
    
    # Test klasörüne kopyala/taşı
    print("Test klasörüne taşınıyor...")
    for i, src_file in enumerate(test_files, 1):
        dst_file = test_dir / src_file.name
        shutil.move(str(src_file), str(dst_file))
        if i % 100 == 0:
            print(f"  {i}/{len(test_files)}")
    print(f"✓ {len(test_files)} resim test klasörüne taşındı")
    print()
    
    print(f"Başarılı! Klasör yapısı:")
    print(f"  {images_dir}/")
    print(f"    ├── train/ ({len(train_files)})")
    print(f"    └── test/ ({len(test_files)})")
    
    return len(train_files), len(test_files)

## Kullanım

In [2]:
# Ayarlar
images_dir = Path("../datasets/dragos/images/simulation")
train_ratio = 0.8  # %80 train, %20 test

# Train/test ayırımını yap
train_count, test_count = split_train_test(
    images_dir=images_dir,
    train_ratio=train_ratio,
    seed=42  # Reproducibility için
)

Toplam resim: 826
Train oranı: 80.0% (660 resim)
Test oranı: 20.0% (166 resim)

Train klasörüne taşınıyor...
  100/660
  200/660
  300/660
  400/660
  500/660
  600/660
✓ 660 resim train klasörüne taşındı

Test klasörüne taşınıyor...
  100/166
✓ 166 resim test klasörüne taşındı

Başarılı! Klasör yapısı:
  ..\datasets\dragos\images\simulation/
    ├── train/ (660)
    └── test/ (166)
